In [6]:
import configparser
from sklearn.model_selection import ParameterGrid
import os
import numpy as np

In [ ]:
def create_directory(folderpath):

    os.makedirs(folderpath)

    os.system('cp /nfs/scistore26/saricgrp/dpersoon/vesicles/code/NIC_clustering/Vesicle.py' + ' ' + folderpath)
    os.system('cp /nfs/scistore26/saricgrp/dpersoon/vesicles/code/NIC_clustering/VesiclePatch.py' + ' ' + folderpath)
    os.system('cp /nfs/scistore26/saricgrp/dpersoon/vesicles/code/NIC_clustering/VesicleInput.py' + ' ' + folderpath)
    os.system('cp /nfs/scistore26/saricgrp/dpersoon/vesicles/code/NIC_clustering/cfg.ini' + ' ' + folderpath)



In [8]:
def create_config(folderpath, base_params):
    cfg = configparser.ConfigParser()
    cfg.optionxform = str
    cfg['_'] = {}
    
    for key in base_params:
        cfg['_'][key] = str(base_params[key])

    f=open(folderpath + '/cfg.ini', 'w', encoding='utf-8')
    cfg.write(f)

In [ ]:
def write_slurm(folderpath, name, string):
    f = open('%s/%s.slurm'%(folderpath,name), 'w')
    f.write('''#!/bin/bash -l
#
#SBATCH --job-name=%s
#SBATCH --output=%s.out
#
#
#SBATCH --ntasks-per-core=1
#
#Define the number of nodes the job should be distributed on
#SBATCH --nodes=1
#
#Total number of CPU cores to be used for the MPI job
#SBATCH --ntasks=4
#
#Define the number of hours the job should run. 
#Maximum runtime is limited to 10 days, ie. 240 hours
#SBATCH --time=24:00:00
#
#Define the amount of RAM used per CPU in GigaBytes
#In distributed memory applications the total amount of RAM 
#used will be:   number of CPUs * memory per CPU
#SBATCH --mem-per-cpu=10G
#
#Pick whether you prefer requeue or not. If you use the --requeue
#option, the requeued job script will start from the beginning, 
#potentially overwriting your previous progress, so be careful.
#For some people the --requeue option might be desired if their
#application will continue from the last state.
#Do not requeue the job in the case it fails.
#SBATCH --no-requeue

# 
##SBATCH --exclude=lemeshko07,bjoern22,bigterra152,lemeshko04,bjoern53,bjoern51,bea82
#for pure MPI jobs the number of threads has to be one
export OMP_NUM_THREADS=1
#
#
#run the respective binary through SLURM's srun
            
module load openmpi/4.1.6
module load netcdf/4.9.2
#module load pnetcdf/1.12.3

#eval "$(micromamba shell hook --shell bash)"
#micromamba activate /nfs/scistore26/saricgrp/dpersoon/micromamba/envs/lammpsenv
            
mpirun -np 4 /nfs/scistore26/saricgrp/dpersoon/lammps/build/lmp -in %s/%s

''' % (name, name, folderpath, string))
    f.close()

In [ ]:
path = '/nfs/scistore26/saricgrp/dpersoon/vesicles/data/NIC_clustering/'
name = 'small_patch'
folderpath = path + name
full_path = folderpath + '/R_8/'

create_directory(full_path)
#create_config(full_path, params)
os.chdir(full_path)
os.system('python VesiclePatch.py')
os.system('python VesicleInput.py')
write_slurm(full_path, name, 'in.local')
os.system('sbatch %s.slurm'%(name))

name = 'small_fraction'
folderpath = path + name
full_path = folderpath + '/x_0.1/'

create_directory(full_path)
#create_config(full_path, params)
os.chdir(full_path)
os.system('python Vesicle.py')
os.system('python VesicleInput.py')
write_slurm(full_path, name, 'in.local')
os.system('sbatch %s.slurm'%(name))